In [ ]:
import torch
import trimesh
import numpy as np
from pytorch3d.structures import Meshes
from pytorch3d.ops import sample_points_from_meshes
from pytorch3d.loss import point_mesh_face_distance

def extract_shell_metal(mesh):
    device = torch.device("mps") # Используем GPU Apple
    
    # Конвертация в PyTorch3D
    verts = torch.tensor(mesh.vertices, dtype=torch.float32, device=device)
    faces = torch.tensor(mesh.faces, dtype=torch.int64, device=device)
    
    pytorch_mesh = Meshes(verts=[verts], faces=[faces])
    
    # Берем центры граней
    face_centers = torch.tensor(mesh.triangles_center, dtype=torch.float32, device=device)
    face_normals = torch.tensor(mesh.face_normals, dtype=torch.float32, device=device)
    
    # Смещаем центры чуть наружу по нормали (эмуляция луча)
    eps = mesh.scale * 1e-4
    query_points = face_centers + face_normals * eps
    
    # Считаем расстояние от этих "смещенных центров" до ближайшей грани меша.
    # Если расстояние > eps/2, значит луч улетел, не задев меш -> Грань видна.
    dist = point_mesh_face_distance(pytorch_mesh, query_points.unsqueeze(0))
    
    # Возвращаем маску видимости
    visible_mask = dist > (eps * 0.1)
    
    return visible_mask.cpu().numpy()
from pytorch3d.io import load_objs_as_meshes
mesh = load_objs_as_meshes(["data/7778_3a9748b3/assembly.obj"])
mask = extract_shell_metal(mesh)

/Users/neonilllai/miniconda3/envs/fusion_env/lib/python3.10/site-packages/pytorch3d/io/obj_io.py:547: UserWarning: No mtl file provided
  warnings.warn("No mtl file provided")


AttributeError: 'Meshes' object has no attribute 'vertices'

In [3]:
import torch
import numpy as np
from pytorch3d.io import load_objs_as_meshes
from pytorch3d.structures import Meshes
from pytorch3d.ops import ray_marching
from pytorch3d.renderer import (
    RayTracing,
    FoVPerspectiveCameras,
    MeshRasterizer,
    RasterizationSettings
)

def extract_visible_faces_pytorch3d(obj_path, device='mps'):
    """
    Чистое решение на PyTorch3D.
    Возвращает маску видимых граней.
    """
    # 1. Загружаем меш напрямую через PyTorch3D
    mesh = load_objs_as_meshes([obj_path], device=device)
    
    # 2. Получаем вершины и грани
    verts = mesh.verts_packed()  # [V, 3]
    faces = mesh.faces_packed()  # [F, 3]
    
    # 3. Вычисляем центры граней
    face_verts = verts[faces]  # [F, 3, 3]
    face_centers = face_verts.mean(dim=1)  # [F, 3]
    
    # 4. Вычисляем нормали граней
    v0 = face_verts[:, 0, :]
    v1 = face_verts[:, 1, :]
    v2 = face_verts[:, 2, :]
    
    # Нормаль = cross(v1-v0, v2-v0)
    edge1 = v1 - v0
    edge2 = v2 - v0
    face_normals = torch.cross(edge1, edge2, dim=1)
    face_normals = face_normals / torch.norm(face_normals, dim=1, keepdim=True)
    
    # 5. Смещаем origins чуть-чуть по нормали
    bbox_size = (verts.max(dim=0)[0] - verts.min(dim=0)[0]).max()
    eps = bbox_size * 1e-4
    ray_origins = face_centers + face_normals * eps  # [F, 3]
    
    # 6. Лучи идут вдоль нормали
    ray_directions = face_normals  # [F, 3]
    
    # 7. Трассировка лучей
    raytracer = RayTracing()
    
    # Добавляем batch dimension [1, F, 3]
    ray_origins_batch = ray_origins.unsqueeze(0)
    ray_directions_batch = ray_directions.unsqueeze(0)
    
    # Трассируем
    # rays_hit: [1, F] bool - попал ли луч в геометрию
    rays_hit = raytracer(
        ray_origins_batch,
        ray_directions_batch,
        mesh
    )
    
    # 8. Видимые грани = те, чей луч НЕ попал в геометрию
    visible_mask = ~rays_hit.squeeze(0)  # [F] bool
    
    return visible_mask.cpu().numpy()

ImportError: cannot import name 'ray_marching' from 'pytorch3d.ops' (/Users/neonilllai/miniconda3/envs/fusion_env/lib/python3.10/site-packages/pytorch3d/ops/__init__.py)